The objective of this notebook is to create reproducible training and evaluation datasets from your published Kaggle dataset.
Our split will be 80% training / 20% testing, using random_state=42 for reproducibility.

In [1]:
# Need to Clone for every Fresh Runtime Session in Colab
!git clone https://github.com/Dgowda11/llama-spam-finetuning.git /content/repo-clone

Cloning into '/content/repo-clone'...
remote: Enumerating objects: 66, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 66 (delta 26), reused 40 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (66/66), 153.80 KiB | 1.79 MiB/s, done.
Resolving deltas: 100% (26/26), done.


In [2]:
# Set Base path and Pull the Recent Changes from the repo to colab
from pathlib import Path
import os

PROJECT_ROOT = Path("/content/repo-clone")

print("Exists:", PROJECT_ROOT.exists())
print("Is directory:", PROJECT_ROOT.is_dir())
print("Git metadata exists:", (PROJECT_ROOT / ".git").exists())

print("\nContents of /content:")
print(os.listdir("/content"))

print("\nRepository directory contents:")
if PROJECT_ROOT.exists():
    print(os.listdir(PROJECT_ROOT))

%cd /content/repo-clone
!git status
!git pull origin main

Exists: True
Is directory: True
Git metadata exists: True

Contents of /content:
['.config', 'repo-clone', 'sample_data']

Repository directory contents:
['notebooks', 'src', 'LICENSE', 'README.md', '.git', '.gitignore']
/content/repo-clone
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
From https://github.com/Dgowda11/llama-spam-finetuning
 * branch            main       -> FETCH_HEAD
Already up to date.


In [3]:
#Import required libraries
import pandas as pd
from sklearn.model_selection import train_test_split

In [4]:
%pip install -q kaggle

In [5]:
KAGGLE_DATASET_ID = "dgowda11/sms-spam-cleaned"

TEST_SIZE = 0.20
RANDOM_STATE = 42

Download the Published Dataset

In [6]:
from pathlib import Path

DATA_DIR = PROJECT_ROOT / "data" / "prepared"
DATA_DIR.mkdir(parents=True, exist_ok=True)

!kaggle datasets download -d dgowda11/sms-spam-cleaned \
    -p /content/sms-spam-download \
    --unzip

Dataset URL: https://www.kaggle.com/datasets/dgowda11/sms-spam-cleaned
License(s): unknown
100% 193k/193k [00:00<00:00, 85.1MB/s]



Verify the Downloaded Dataset

In [7]:
df = pd.read_csv("/content/sms-spam-download/sms_spam_cleaned.csv")
df.shape
df.head(5)

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


Verify the Downloaded Dataset from the observation we had in the Data Exploration Stage 

In [8]:
assert df.shape == (5169, 2), "Unexpected dataset shape"

assert df["label"].isin(["ham", "spam"]).all(), "Unexpected labels found"

assert df["message"].notnull().all(), "Missing text values found"

assert df["message"].duplicated().sum() == 0, "Duplicate rows found"

assert df["message"].str.strip().ne("").all(), "Empty message values found"

print("All dataset Integrity checks passed successfully.")

All dataset Integrity checks passed successfully.


Stratified Train/Test Split

Why Stratified Splitting?
Our dataset is imbalanced:
Class	    Percentage
Ham	           ~87%
Spam	       ~13%

A normal random split might slightly alter the class distribution. Stratified splitting preserves approximately the same class proportions in both training and testing datasets.

In [9]:
# Configure the Split parameters 
from sklearn.model_selection import train_test_split

TEST_SIZE = 0.20 # → 80% training, 20% testing
RANDOM_STATE = 42

In [10]:
# Perform the Split 
train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["label"]
)

train_df  = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train shape :", train_df.shape)
print("Test shape :", test_df.shape)

Train shape : (4135, 2)
Test shape : (1034, 2)


Validate the Split 

In [11]:
# Compare class distributions
print("Original Distributions")
print(df["label"].value_counts(normalize=True))

print("Train Distributions")
print(train_df["label"].value_counts(normalize=True))

print("Test Distributions")
print(test_df["label"].value_counts(normalize=True))

Original Distributions
label
ham     0.87367
spam    0.12633
Name: proportion, dtype: float64
Train Distributions
label
ham     0.873761
spam    0.126239
Name: proportion, dtype: float64
Test Distributions
label
ham     0.873308
spam    0.126692
Name: proportion, dtype: float64


In [17]:
    # Verify split integrity

assert len(train_df) + len(test_df) == len(df)


assert train_df["label"].value_counts().to_dict() == {
    "ham": 3613,
    "spam": 522
}

assert test_df["label"].value_counts().to_dict() == {
    "ham": 903,
    "spam": 131
}

print("Train/Test split validation passed!")

Train/Test split validation passed!


Leakage check between Train and Test Dataset

In [18]:
assert set(train_df["message"]).isdisjoint(
    set(test_df["message"])
)

print("No message leakage between train and test!")

No message leakage between train and test!


Save Prepared Datasets

In [19]:
from pathlib import Path

DATA_DIR = PROJECT_ROOT / "data" / "prepared"
DATA_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

train_df.to_csv(TRAIN_PATH, index=False)
test_df.to_csv(TEST_PATH, index=False)

print("Training dataset saved:", TRAIN_PATH)
print("Testing dataset saved:", TEST_PATH)

Training dataset saved: /content/repo-clone/data/prepared/train.csv
Testing dataset saved: /content/repo-clone/data/prepared/test.csv


Upload the Train/Test Split to Kaggle Dataset with new version

In [36]:
from pathlib import Path
import shutil

REPO_DATA_DIR = Path("/content/repo-clone/data/prepared")
KAGGLE_UPLOAD_DIR = Path("/content/kaggle-splits")

KAGGLE_UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Copy train and test splits
shutil.copy(REPO_DATA_DIR / "train.csv", KAGGLE_UPLOAD_DIR / "train.csv")
shutil.copy(REPO_DATA_DIR / "test.csv", KAGGLE_UPLOAD_DIR / "test.csv")

print(list(KAGGLE_UPLOAD_DIR.iterdir()))

[PosixPath('/content/kaggle-splits/train.csv'), PosixPath('/content/kaggle-splits/test.csv')]


In [39]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Enter your Kaggle API token: ")

In [ ]:
# Download the existing dataset metadata from Kaggle
!kaggle datasets metadata dgowda11/sms-spam-cleaned \
    -p /content/kaggle-splits

Downloaded metadata to /content/kaggle-splits/dataset-metadata.json


In [48]:
!kaggle datasets version \
    -p /content/kaggle-splits \
    -m "Added stratified train and test splits"

Starting upload for file train.csv
100% 343k/343k [00:00<00:00, 616kB/s]
Upload successful: train.csv (343KB)
Starting upload for file test.csv
100% 88.3k/88.3k [00:00<00:00, 171kB/s]
Upload successful: test.csv (88KB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/dgowda11/sms-spam-cleaned
